<a href="https://colab.research.google.com/github/LuizFellipiFreire25/Projeto-ECAA08/blob/main/etapa-02-grafos/15%20-%20Simulacao%20de%20Vazamentos%20e%20Desvio%20Automatico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 15: Simulação de Obstrução de Corredor e Desvio Automático em Malha Fechada

## 1. Fundamentos Matemáticos: Reconfiguração Dinâmica de Grafos em Tempo Real

No sistema **SCADA-Core / Visão-AGV**, a detecção de um obstáculo no corredor $e = (u, v)$ dispara a reconfiguração topológica instantânea:

$$W(u, v) \leftarrow \infty$$

A rota de desvio é calculada via Dijkstra em tempo real:

$$\vec{P}_{\text{novo}} = \text{Dijkstra}(G_{\text{reconfigurado}}, s, t)$$

In [2]:
import heapq
import time
from typing import Dict, List, Tuple, Optional

class SimuladorDesvioAGV:
    """
    Simulador de Roteamento em Malha Fechada com Recálculo Dinâmico
    para Reação a Obstáculos no Galpão.
    """
    def __init__(self):
        self.estacoes: set = set()
        self.adj: Dict[str, List[Tuple[str, float]]] = {}

    def adicionar_corredor(self, origem: str, destino: str, custo_m: float):
        self.estacoes.add(origem)
        self.estacoes.add(destino)
        if origem not in self.adj:
            self.adj[origem] = []
        self.adj[origem].append((destino, custo_m))

    def injetar_obstrucao(self, origem: str, destino: str):
        """Simula a detecção de obstáculo via LiDAR/RFID definindo W(u,v) = inf."""
        if origem in self.adj:
            self.adj[origem] = [(v, float('inf') if v == destino else c) for v, c in self.adj[origem]]

    def calcular_dijkstra(self, origem: str, destino: str) -> Tuple[Optional[List[str]], float, float]:
        """Calcula o caminho mínimo e retorna (caminho, distancia_m, tempo_execucao_ms)."""
        t_inicio = time.perf_counter()

        distancias = {node: float('inf') for node in self.estacoes}
        predecessores = {node: None for node in self.estacoes}
        distancias[origem] = 0.0
        min_heap = [(0.0, origem)]

        while min_heap:
            dist_atual, u = heapq.heappop(min_heap)

            if dist_atual > distancias[u]:
                continue
            if u == destino:
                break

            for vizinho, peso in self.adj.get(u, []):
                if peso == float('inf'):
                    continue
                dist_alt = dist_atual + peso
                if dist_alt < distancias[vizinho]:
                    distancias[vizinho] = dist_alt
                    predecessores[vizinho] = u
                    heapq.heappush(min_heap, (dist_alt, vizinho))

        t_fim = time.perf_counter()
        tempo_ms = (t_fim - t_inicio) * 1000.0

        if distancias[destino] == float('inf'):
            return None, float('inf'), tempo_ms

        caminho = []
        passo = destino
        while passo is not None:
            caminho.append(passo)
            passo = predecessores[passo]
        caminho.reverse()

        return caminho, distancias[destino], tempo_ms

## 2. Teste de Estresse e Resposta Dinâmica em Milissegundos
Execução da simulação com injeção de obstáculo no corredor direto `AMO-301 -> DEP-401`.

In [3]:
simulador = SimuladorDesvioAGV()

corredores = [
    ("ST-01", "DOC-101", 10.0),
    ("ST-01", "ALM-201", 12.0),
    ("DOC-101", "ALM-201", 15.0),
    ("ALM-201", "AMO-301", 20.0),
    ("AMO-301", "R-101", 18.0),
    ("AMO-301", "DEP-401", 14.0),
    ("R-101", "DEP-401", 25.0),
    ("DEP-401", "ST-01", 30.0)
]

for u, v, w in corredores:
    simulador.adicionar_corredor(u, v, w)

# 1. Trajeto Nominal
rota_nom, dist_nom, t_nom = simulador.calcular_dijkstra("ST-01", "DEP-401")
print(f"[NOMINAL] Rota: {' -> '.join(rota_nom)} | Distância: {dist_nom:.1f}m | Tempo: {t_nom:.4f} ms")

# 2. Injeção de Bloqueio
print("\n[ALERTA SCADA] Bloqueio detectado no corredor AMO-301 -> DEP-401!")
simulador.injetar_obstrucao("AMO-301", "DEP-401")

# 3. Trajeto com Desvio
rota_desvio, dist_desvio, t_desvio = simulador.calcular_dijkstra("ST-01", "DEP-401")
print(f"[DESVIO] Rota Alternativa: {' -> '.join(rota_desvio)} | Nova Distância: {dist_desvio:.1f}m | Tempo de Recálculo: {t_desvio:.4f} ms")

[NOMINAL] Rota: ST-01 -> ALM-201 -> AMO-301 -> DEP-401 | Distância: 46.0m | Tempo: 0.0257 ms

[ALERTA SCADA] Bloqueio detectado no corredor AMO-301 -> DEP-401!
[DESVIO] Rota Alternativa: ST-01 -> ALM-201 -> AMO-301 -> R-101 -> DEP-401 | Nova Distância: 75.0m | Tempo de Recálculo: 0.0147 ms
